# z016 - Undersampling de CONTINUA (aprendizaje para la segunda competencia)

**Idea:** entrenar con **todos los BAJA+1 y BAJA+2** y solo una parte al azar de los CONTINUA (100%, 50%, 20%).
Se mide cuánto se pierde de ganancia y cuánto más rápido entrena.

Ojo: los hiperparámetros del trial 34 se ajustaron con el 100% → con menos datos no son los óptimos,
así que el resultado está sesgado en contra del undersampling.
Trial 34, 538 árboles, 10 semillas, train 202104, valid 202106. Se corta por cantidad (top N).

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331,   # las tuyas
             111119, 222227, 333331, 444443, 555557]  # 5 más (las mismas del ensamble)
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Funciones de evaluación ---
def metricas(prob, mascara=None):
    # Ganancia acumulada ordenando de mayor a menor probabilidad (opcional: solo una parte de los clientes)
    g = ganancia_cliente if mascara is None else ganancia_cliente[mascara]
    p = prob if mascara is None else prob[mascara]
    acum = np.cumsum(g[np.argsort(p)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[min(11_999, len(acum) - 1)] / 1e6

def resumen(probs_por_version):
    """probs_por_version = {version: [prob_semilla1, ..., prob_semilla5]}"""
    # a) Por semilla: ganancia máxima, mejor N y top 12.000
    filas = []
    for v, lista in probs_por_version.items():
        for s, pr in zip(SEMILLAS, lista):
            g, n, g12 = metricas(pr)
            filas.append({"version": v, "semilla": s, "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
    por_semilla = pd.DataFrame(filas)
    tabla = por_semilla.groupby("version", sort=False)[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)

    # b) Shuffle Split del test: ensamble de las 5 semillas evaluado en 50 particiones del 50% de 202106
    rng = np.random.default_rng(SEMILLAS[0])
    ens = {v: np.mean(lista, axis=0) for v, lista in probs_por_version.items()}
    part = []
    for i in range(N_PARTICIONES):
        mascara = rng.random(len(ganancia_cliente)) < 0.5
        part.append({v: metricas(pr, mascara)[0] for v, pr in ens.items()})
    part = pd.DataFrame(part)
    tabla[("shuffle_split", "mean")] = part.mean().round(1)
    tabla[("shuffle_split", "std")] = part.std().round(1)
    tabla[("shuffle_split", "veces_mejor")] = part.idxmax(axis=1).value_counts().reindex(tabla.index).fillna(0).astype(int)
    tabla[("ensamble_5", "ganancia_max_M")] = [round(metricas(ens[v])[0], 1) for v in tabla.index]
    return por_semilla, tabla

def anotar(tabla, prefijo, meses_train, extra=""):
    ruta_res = "resultados_experimentos.csv"
    res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
    res = res[~res["descripcion"].str.startswith(prefijo)]   # sin duplicar si se vuelve a correr
    for v, r in tabla.iterrows():
        nueva = {
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"{prefijo} {v}: prom. {len(SEMILLAS)} semillas; shuffle split {r[('shuffle_split', 'mean')]} "
                            f"(mejor en {r[('shuffle_split', 'veces_mejor')]}/{N_PARTICIONES}){extra}"),
            "meses_train": str(meses_train), "mes_valid": MES_VALID, "semilla": str(SEMILLAS),
            "params": str(params), "mejor_n": int(r[("mejor_n", "mean")]),
            "ganancia_max_M": r[("ganancia_max_M", "mean")], "ganancia_top12000_M": r[("top12000_M", "mean")],
        }
        res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
    res["n_enviados"] = res["n_enviados"].astype("Int64")
    res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
    print("Anotado en", ruta_res)

In [4]:
# --- 4. Entrenar con distintos % de CONTINUA ---
import time
train_total = df[df["foto_mes"] == 202104]
probs, tiempos = {}, {}
for pct in [1.0, 0.5, 0.2]:
    version = f"{int(pct * 100)}% de CONTINUA"
    probs[version], t = [], []
    for s in SEMILLAS:
        rng = np.random.default_rng(s)
        quedan = (train_total["clase_ternaria"] != "CONTINUA") | (rng.random(len(train_total)) < pct)
        train = train_total[quedan]
        inicio = time.time()
        modelo = lgb.train({**params, "seed": s}, lgb.Dataset(train[variables], label=train["clase_binaria"]),
                           num_boost_round=ARBOLES)
        t.append(time.time() - inicio)
        probs[version].append(modelo.predict(valid[variables]))
    tiempos[version] = round(np.mean(t), 1)
    print(version, "| filas:", len(train), "| segundos por modelo:", tiempos[version])

100% de CONTINUA | filas: 163284 | segundos por modelo: 26.6


50% de CONTINUA | filas: 82732 | segundos por modelo: 15.4


20% de CONTINUA | filas: 34376 | segundos por modelo: 8.1


In [5]:
# --- 5. Comparación ---
por_semilla, tabla = resumen(probs)
print(por_semilla.pivot(index="semilla", columns="version", values="ganancia_max_M"))
tabla

version  100% de CONTINUA  20% de CONTINUA  50% de CONTINUA
semilla                                                    
111119              396.8            388.8            406.5
222227              396.8            393.7            400.0
333331              403.6            393.1            405.7
444443              396.9            398.6            396.6
501419              400.5            377.2            392.7
555557              401.0            387.6            390.4
623353              397.2            392.1            385.2
786349              401.4            405.3            394.9
834569              397.2            397.8            389.5
946331              401.8            385.7            396.2


ganancia_max_M      top12000_M       mejor_n          \
                           mean  std       mean  std     mean     std   
version                                                                 
100% de CONTINUA          399.3  2.6      391.7  3.7  13230.9  1224.2   
50% de CONTINUA           395.8  6.9      389.6  8.7  13464.3  1472.1   
20% de CONTINUA           392.0  7.8      381.7  7.0  13437.7  1013.3   

                 shuffle_split                       ensamble_5  
                          mean   std veces_mejor ganancia_max_M  
version                                                          
100% de CONTINUA         198.9  11.8          43          400.5  
50% de CONTINUA          196.8  11.3           6          396.9  
20% de CONTINUA          194.2  11.5           1          391.3

In [6]:
# --- 6. Anotar ---
anotar(tabla, "z016 undersampling", [202104])

Anotado en resultados_experimentos.csv


## Conclusiones (08/10)
| CONTINUA | Filas | Segundos por modelo | Ganancia |
|---|---|---|---|
| 100% | 163.284 | 26,6 | 399,3 |
| 50% | 82.732 | 15,4 | 395,8 (−3,5) |
| 20% | 34.376 | 8,1 | 392,0 (−7,3) |
- Con el 20% entrena **3,3 veces más rápido** y pierde ~7 M (con hiperparámetros ajustados al 100%, sesgado en contra).
- Para la 2da competencia: el undersampling es viable si hace falta velocidad.